In [ ]:
# 导入本地包
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# import seaborn as sns
import torch
import torch.nn as nn
import scipy
from scipy.integrate import solve_ivp
import argparse
import gymnasium as gym
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3 import PPO
import os

# 设置命令行输入的参数解析器
try:
    parser = argparse.ArgumentParser()
    # 模型参数导出路径
    parser.add_argument('--export_path', type=str, default='model', help='文件夹路径，不包含/')
    # 预训练PPO模型加载路径
    parser.add_argument('--model_path', type=str, default='Training/Saved Models/best_model', help='PPO模型路径，不包含.zip扩展名')
    # 每个回合允许的最大步数
    parser.add_argument('--max_episode_step', type=int, default=2000, help='每个回合允许的最大步数')
    args = parser.parse_args()
except SystemExit:
    # 对于Jupyter环境，设置默认值
    class Args:
        export_path = 'Text_Results'  # 改为其他文件夹名
        model_path = 'Training/Saved Models/test_model'
        max_episode_step = 2000
    args = Args()

# 提取解析的参数以便使用
export_path = args.export_path
model_path = args.model_path
max_episode_step = args.max_episode_step

# 创建导出目录
os.makedirs(export_path, exist_ok=True)

# 注册自定义环境（使用CartPoleTest避免与内置CartPole冲突）
gym.register(
    id='CartPoleTest',
    entry_point='myCartpoleF:myCartPoleEnvF',  # 自定义环境位置
    reward_threshold=1950,  # 环境完成的奖励阈值
    max_episode_steps=max_episode_step  # 每个回合的最大步数
)

env = DummyVecEnv([lambda: gym.make('CartPoleTest', render_mode='rgb_array')])

# 加载PPO模型（从指定的模型文件路径加载）
model = PPO.load(path=model_path + '.zip', env=env)

print(f"模型结构: {model.policy}\n\n")

# 获取模型参数
model_param = model.get_parameters()
model_policy = model_param['policy']
print("可用的策略参数:")
for key in model_policy.keys():
    print(f"  {key}: {model_policy[key].shape}")

print("\n提取Actor网络参数...")

# 查找并保存Actor网络参数
for param_tensor in model_policy.keys():
    # 在PPO策略中查找Actor相关参数
    if any(keyword in param_tensor for keyword in ['action_net', 'mlp_extractor.policy_net', 'features_extractor']):
        print(param_tensor + ':' + str(model_policy[param_tensor].shape))
        
        # 保存每个参数到文本文件（遵循老师的命名约定）
        np.savetxt(export_path + '/[PPO_MODEL]' + param_tensor + '.txt',
                   model_policy[param_tensor],
                   delimiter=',')

# 按照老师的命名约定提取网络权重和偏置
print("\n以MATLAB兼容格式提取权重和偏置...")

# 在PPO策略中查找实际的层名称
actor_params = {}
for key, value in model_policy.items():
    if 'action_net' in key or 'mlp_extractor.policy_net' in key or 'features_extractor' in key:
        actor_params[key] = value

# 打印可用参数以供手动分配
print("可用的Actor参数:")
for key, value in actor_params.items():
    print(f"  {key}: 形状 {value.shape}")

# 尝试自动提取层（根据实际PPO结构调整）
try:
    # 根据你的实际PPO模型结构进行调整
    
    # 第一个隐藏层
    if 'mlp_extractor.policy_net.0.weight' in model_policy:
        W0 = np.array(model_policy['mlp_extractor.policy_net.0.weight'], dtype=np.float32)
        b0 = np.array(model_policy['mlp_extractor.policy_net.0.bias'], dtype=np.float32)
        print(f"W0 形状: {W0.shape}, b0 形状: {b0.shape}")
    
    # 第二个隐藏层
    if 'mlp_extractor.policy_net.2.weight' in model_policy:
        W1 = np.array(model_policy['mlp_extractor.policy_net.2.weight'], dtype=np.float32)
        b1 = np.array(model_policy['mlp_extractor.policy_net.2.bias'], dtype=np.float32)
        print(f"W1 形状: {W1.shape}, b1 形状: {b1.shape}")
    
    # 输出动作层
    if 'action_net.weight' in model_policy:
        W_out = np.array(model_policy['action_net.weight'], dtype=np.float32)
        b_out = np.array(model_policy['action_net.bias'], dtype=np.float32)
        print(f"W_out 形状: {W_out.shape}, b_out 形状: {b_out.shape}")
    
    # 保存个别权重矩阵
    if 'W0' in locals():
        np.savetxt(export_path + '/W0.txt', W0, delimiter=',')
        np.savetxt(export_path + '/b0.txt', b0, delimiter=',')
    if 'W1' in locals():
        np.savetxt(export_path + '/W1.txt', W1, delimiter=',')
        np.savetxt(export_path + '/b1.txt', b1, delimiter=',')
    if 'W_out' in locals():
        np.savetxt(export_path + '/W_out.txt', W_out, delimiter=',')
        np.savetxt(export_path + '/b_out.txt', b_out, delimiter=',')
    
    print(f"\n模型参数已导出到: {export_path}/")
    print("已保存文件: W0.txt, b0.txt, W1.txt, b1.txt, W_out.txt, b_out.txt")
    
except Exception as e:
    print(f"自动提取错误: {e}")
    print("请检查参数名称并相应调整代码。")

usage: ipykernel_launcher.py [-h] [--export_path EXPORT_PATH]
                             [--model_path MODEL_PATH]
                             [--max_episode_step MAX_EPISODE_STEP]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\86133\AppData\Roaming\jupyter\runtime\kernel-v3934f03f2cdb1d6ea9f4085ab08519739742cac84.json


模型结构: ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=1, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)


可用的策略参数:
  log_std: torch.Size([1])
  mlp_extractor.policy_net.0.weight: torch.Size([64, 4])
  mlp_extractor.policy_net.0.bias

C:\Users\86133\AppData\Local\Temp\ipykernel_15344\2688598135.py:98: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  W0 = np.array(model_policy['mlp_extractor.policy_net.0.weight'], dtype=np.float32)
C:\Users\86133\AppData\Local\Temp\ipykernel_15344\2688598135.py:99: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  b0 = np.array(model_policy['mlp_extractor.policy_net.0.bias'], dtype=np.float32)
C:\Users\86133\AppData\Local\Temp\ipykernel_15344\2688598135.py:104: DeprecationWarnin